# 📊 A股因子选股入门教程

## 什么是因子选股？

**因子**是能解释股票收益的某种特征。比如：
- 低估值的股票是否长期跑赢高估值？ → **估值因子**
- 高ROE的公司是否涨得更好？ → **质量因子**
- 过去涨得好的股票是否继续涨？ → **动量因子**

**因子选股**就是将多个有效因子合起来，给每只股票打分，选得分最高的构建组合。

---

In [ ]:
# 导入基础库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文支持
plt.rcParams['axes.unicode_minus'] = False

print('✅ 导入成功')

---
## Step 1: 获取数据

我们使用 **AKShare** (免费开源) 获取A股数据。

> 如果第一次运行，需要安装：`pip install akshare`

In [ ]:
# 导入我们的模块
import sys
sys.path.append('..')

from data_fetcher import AStockDataFetcher
from config import BACKTEST_CONFIG, FACTOR_CONFIG

fetcher = AStockDataFetcher()

# 获取股票列表
stocks = fetcher.get_stock_list()
print(f'A股总数: {len(stocks)}')
stocks.head(10)

### 获取行情数据

先取少量股票做演示。实际你可以去掉 `[:10]` 限制。

In [ ]:
# 取前10只股票演示
sample_codes = stocks['stock_code'].tolist()[:10]
print(f'样例股票: {sample_codes}')

price_df = fetcher.get_all_daily_price('2024-01-01', '2024-12-31', sample_codes)
print(f'数据量: {price_df.shape[0]} 行 × {price_df.shape[1]} 列')
price_df.head()

---
## Step 2: 计算因子

以 **PE_TTM (市盈率)** 为例。

> PE = 市值 / 净利润
> 
> 低PE通常意味着估值偏低，可能被低估

In [ ]:
from factors.fundamental import PE_TTM

pe_factor = PE_TTM()
print(f'因子: {pe_factor.name} | 类别: {pe_factor.category} | 方向: {"多" if pe_factor.direction == 1 else "空"}')

# 计算因子值
factor_values = pe_factor.calculate(price_df)
print(f'\n因子值 (最新截面):')
factor_values.sort_values().head(10)

---
## Step 3: 因子分析 — IC (Information Coefficient)

**IC = 因子值与未来收益的相关系数**

- **IC > 0**: 因子值越大，未来收益越高
- **IC < 0**: 因子值越小，未来收益越高 (负向因子)
- **|IC| > 0.02**: 通常认为有选股能力

**Rank IC**: 用排序计算相关系数，对极端值不敏感，更稳健

In [ ]:
from analyzer import FactorAnalyzer

analyzer = FactorAnalyzer(group_num=10)

# 构建因子矩阵 (多期)
from factor_pipeline import FactorPipeline
pipeline = FactorPipeline()

# 用演示数据
factor_matrix = pipeline._build_factor_matrix(pe_factor, price_df)
forward_ret = pipeline._calc_forward_returns(price_df, hold_days=20)

print(f'因子矩阵: {factor_matrix.shape} (日期×股票)')
print(f'收益矩阵: {forward_ret.shape} (日期×股票)')
factor_matrix.head()

In [ ]:
# IC分析
ic_series = analyzer.calc_ic(factor_matrix, forward_ret)
rank_ic = analyzer.calc_rank_ic(factor_matrix, forward_ret)

print(f'IC均值:     {ic_series.mean():.4f}')
print(f'IC标准差:   {ic_series.std():.4f}')
print(f'IC信息比:   {ic_series.mean()/ic_series.std():.4f}')
print(f'IC>0比例:   {(ic_series>0).mean():.2%}')
print(f'Rank IC均值:{rank_ic.mean():.4f}')

# 画IC时间序列
fig = analyzer.plot_ic_series(ic_series, f'{pe_factor.name} IC时序')
plt.show()

In [ ]:
# 分组收益分析
group_ret = analyzer.calc_group_returns(factor_matrix, forward_ret)

fig = analyzer.plot_group_cum_return(group_ret, f'{pe_factor.name} 分组累计收益')
plt.show()

print(f'多空组合累计收益: {(1 + group_ret[10]).prod() - 1:.2%}')

### 解读分组收益图

- 如果 **Group 1 (因子值最小)** 跑赢 **Group 10 (因子值最大)** → 因子有效，方向为负
- 如果分组收益**单调递增/递减** → 因子区分度好
- **Long-Short 曲线向上** → 多空策略赚钱

---
## Step 4: 多因子合成

单因子不稳定，多因子组合更鲁棒。

我们使用 **等权打分法**:
1. 对每个因子，将股票转化为 **0~1 分**
2. 将所有因子得分**取平均**
3. 选总分最高的股票

In [ ]:
from multi_factor import MultiFactorModel

model = MultiFactorModel()

# 注册两个因子: PE (负向) + ROE (正向)
from factors.fundamental import ROE
from factors.momentum import Momentum1M

factor_pe = PE_TTM()
factor_roe = ROE()
factor_mom = Momentum1M()

# 构建各因子的矩阵
for factor_obj, name in [(factor_pe, 'pe_ttm'), (factor_roe, 'roe'), (factor_mom, 'momentum_1m')]:
    matrix = pipeline._build_factor_matrix(factor_obj, price_df)
    if not matrix.empty:
        values_dict = {d: matrix.loc[d] for d in matrix.index}
        model.add_factor(name, values_dict, direction=factor_obj.direction, category=factor_obj.category)
        print(f'✅ 注册因子: {factor_obj.name}')

# 等权合成
combined = model.combine_equal_weight()
print(f'\n综合评分矩阵: {combined.shape}')
combined.head()

In [ ]:
# 选股
selections = model.select_top(combined, top_n=5)

for date, stocks in list(selections.items())[:3]:
    print(f'{date}: {stocks}')

---
## Step 5: 全流程一键运行

如果数据量大，可以直接用命令行方式：

```bash
python factor_pipeline.py --mode full_pipeline
```

或者测试单个因子：

```bash
python factor_pipeline.py --mode test_single --factor roe
```

---
## 扩展学习

掌握基础后，可以深入：

| 进阶方向 | 内容 |
|---------|------|
| **因子组合** | 用IC加权替代等权，动态调整因子权重 |
| **风险模型** | Barra模型，行业中性化，市值中性化 |
| **机器学习因子** | 用XGBoost/LSTM挖掘非线性因子 |
| **另类数据** | 新闻舆情、机构持仓、龙虎榜数据 |
| **算法交易** | 结合vWAP/TWAP降低冲击成本 |

---
> 提示: 策略表现需要长期验证，单期表现不能说明问题。